# Spatial graph representations

> **Notebook role:** production-style conversion of intranuclear organization into graph features and temporal tracks.

![Feature, graph and learned representations](../assets/diagrams/representations.svg)

## 1. Detect dense regions and boundary landmarks

The graph workflow normalizes intensity inside the nuclear mask, detects dense
regions, optionally splits them around peaks and samples the nuclear boundary.

In [ ]:
from pathlib import Path
import numpy as np

from nuclear_imaging_core.graph.dense_regions import DenseRegionConfig, segment_dense_regions_frame

ordered_frames = ("pre", "ref", "dec", "fin")
frame_archive = np.load(Path("data/ordered_nuclear_frames.npz"))
frames = {name: frame_archive[name] for name in ordered_frames}
graph_config = DenseRegionConfig(
    min_region_size_px=20,
    gaussian_sigma_px=1.0,
    with_boundary_nodes=True,
    boundary_num_points=64,
)
frame_result = segment_dense_regions_frame(frames["ref"], "ref", graph_config)
frame_result.node_table.head()

## 2. Build typed edges and graph descriptors

Peak-to-peak, peak-to-boundary and boundary-to-boundary edges retain different
types. Distances are normalized by nuclear geometry before aggregation.

In [ ]:
from nuclear_imaging_core.graph.graph_build import build_frame_graph_bundle

bundle = build_frame_graph_bundle(
    frame_result,
    "reference",
    peak_peak_distance_threshold_norm=0.25,
    peak_boundary_distance_threshold_norm=0.25,
)
bundle.node_table.head(), bundle.edge_table.head(), bundle.graph_attrs

## 3. Track corresponding nodes across ordered frames

Matching uses normalized position and size so structural changes can be
summarized alongside image- and feature-based trajectories.

In [ ]:
from nuclear_imaging_core.graph.node_tracking import TrackingConfig, track_ordered_nodes

nodes_by_frame = {
    name: segment_dense_regions_frame(frames[name], name, graph_config).node_table
    for name in ordered_frames
}
tracking = TrackingConfig(w_position=1.0, w_size=1.0, candidate_radius_px=20.0)
tracks = track_ordered_nodes(nodes_by_frame, frame_order=ordered_frames, cfg=tracking)
tracks["matches_by_frame"]["fin"].head()

## Representative result

Graph descriptors and classical object measurements can be embedded together to
reveal continuous structure across experimental conditions.

![Feature-space embedding](../assets/results/feature-embedding.png)